In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchtext.datasets import IMDB
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence


OSError: /home2/liyang/micromamba/envs/fl310/lib/python3.10/site-packages/torchtext/lib/libtorchtext.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs

In [ ]:
# Tokenizer
tokenizer = get_tokenizer("basic_english")

# Function to yield tokens from dataset
def yield_tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text)

# Build vocabulary from IMDB training dataset
train_iter = IMDB(split='train')
vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=["<PAD>", "<UNK>"])
vocab.set_default_index(vocab["<UNK>"])  # Handle unknown words

# Convert text to tensor
def text_pipeline(text):
    return vocab(tokenizer(text))

# Label conversion
label_pipeline = lambda x: 1 if x == "pos" else 0


In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, split):
        self.data = list(IMDB(split=split))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        label, text = self.data[idx]
        return torch.tensor(label_pipeline(label)), torch.tensor(text_pipeline(text))

# Collate function for padding
def collate_batch(batch):
    labels, texts = zip(*batch)
    labels = torch.tensor(labels, dtype=torch.long)
    texts = pad_sequence(texts, padding_value=vocab["<PAD>"], batch_first=True)
    return labels, texts

# DataLoaders
batch_size = 32
train_dataset = IMDBDataset(split='train')
test_dataset = IMDBDataset(split='test')

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collate_batch)


In [ ]:
class DNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super(DNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=vocab["<PAD>"])
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.5)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, text):
        embedded = self.embedding(text).mean(dim=1)  # Average embeddings over sequence
        x = self.fc1(embedded)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return self.softmax(x)

# Model parameters
vocab_size = len(vocab)
embed_dim = 128
hidden_dim = 256
output_dim = 2  # Binary classification

model = DNNClassifier(vocab_size, embed_dim, hidden_dim, output_dim)


In [ ]:
# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
num_epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for labels, texts in train_loader:
        labels, texts = labels.to(device), texts.to(device)

        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}")


In [ ]:
from sklearn.metrics import accuracy_score

model.eval()
predictions, true_labels = [], []

with torch.no_grad():
    for labels, texts in test_loader:
        labels, texts = labels.to(device), texts.to(device)
        outputs = model(texts)
        _, preds = torch.max(outputs, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(true_labels, predictions)
print(f"Test Accuracy: {accuracy:.4f}")
